# Smart Agriculture: Starter Notebook

**DSU Hackathon, October 17–18, 2026**

This notebook loads the master dataset, walks through one research question, and leaves space for your own analysis.

One row = one county, one crop, one year. 4 states, 4 crops, 26 years, 21 fields.

Read [`data/data_dictionary.md`](../data/data_dictionary.md) before you start — it explains every field, every NaN, and every known limitation.

In [1]:
# Run this cell once if you're in Google Colab
# !git clone <repo-url>
# %cd DSU_HACKATHON/notebooks
# !pip install -q -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px
from src.data_loader import load_master

df = load_master()
print(f"{len(df):,} rows × {len(df.columns)} columns")
df.head(15)

In [ ]:
# What's in the dataset?
print("States:", sorted(df["state_alpha"].unique()))
print("Crops: ", sorted(df["crop"].unique()))
print("Years: ", df["year"].min(), "–", df["year"].max())
print()

# Rows per state and crop
pd.crosstab(df["state_alpha"], df["crop"], margins=True)

---
## Worked example: the 2012 Iowa drought

In the summer of 2012, the worst drought in decades hit the Corn Belt. This walkthrough is about the mechanics such as filter, aggregate, plot - with 2012 Iowa corn as the subject. What the numbers say, you can read off the charts

In [ ]:
# Filter to Iowa corn with reported yields
ia_corn = df[
    (df["state_alpha"] == "IA")
    & (df["crop"] == "CORN")
    & (df["yield_status"] == "reported")
]

# State average by year
yearly = ia_corn.groupby("year", as_index=False).agg(
    mean_yield=("yield_per_acre", "mean"),
    mean_dsci=("mean_dsci", "mean"),
    mean_anomaly=("yield_anomaly_pct", "mean"),
)

fig = px.bar(
    yearly, x="year", y="mean_yield",
    title="Iowa Corn: Mean County Yield by Year",
    labels={"mean_yield": "Yield (bu/acre)", "year": "Year"},
)
fig.add_vrect(x0=2011.5, x1=2012.5, fillcolor="red", opacity=0.15,
              annotation_text="2012", annotation_position="top left")
fig.show()

In [ ]:
# How bad was the drought? Compare DSCI (Drought Severity and Coverage Index)
# 0 = no drought, 500 = entire county in D4 (exceptional) all season

fig = px.bar(
    yearly, x="year", y="mean_dsci",
    title="Iowa Corn: Mean Growing-Season Drought Intensity",
    labels={"mean_dsci": "Mean DSCI", "year": "Year"},
)
fig.add_vrect(x0=2011.5, x1=2012.5, fillcolor="red", opacity=0.15,
              annotation_text="2012", annotation_position="top left")
fig.show()

In [ ]:
# County-level scatter: does higher drought intensity predict worse yield anomaly?
# yield_anomaly_pct removes the long-term trend, so we're comparing to each county's own baseline

reported = ia_corn[ia_corn["yield_anomaly_pct"].notna()].copy()

fig = px.scatter(
    reported, x="mean_dsci", y="yield_anomaly_pct",
    color=reported["year"].eq(2012).map({True: "2012", False: "Other years"}),
    opacity=0.5,
    title="Iowa Corn: Yield Anomaly vs Drought Intensity (every county-year)",
    labels={
        "mean_dsci": "Mean DSCI (higher = worse drought)",
        "yield_anomaly_pct": "Yield anomaly (%)",
        "color": "",
    },
)
fig.show()

**What this walkthrough covered:**

- Filtering to one state, one crop, and `yield_status == "reported"`
- Aggregating county rows into a yearly state average with `groupby().agg()`
- Reading one year against its neighbors, then dropping back to county level to test whether a pattern holds beyond that single year

One field worth understanding before you go further: `yield_anomaly_pct` measures each county against its own long-term trend, so it separates *a bad year* from *a low-yielding county*. See the data dictionary for how it's computed.

---

## Your turn

Three places to start. Each names a question and the fields that bear on it — the approach is yours. Pick one, combine them, or go somewhere else entirely.

### 1. Does soil buffer drought?

Soils differ in how much water they hold for a crop to draw on. Relevant fields: `aws_100cm_mm`, `droughty_pct`, `mean_dsci`, `weeks_in_d2_plus`, `yield_anomaly_pct`. How you define a drought year and what would count as evidence of buffering — is your call.

In [ ]:
# Your analysis here


### 2. Corn vs sorghum in Nebraska

Sorghum is often described as drought-tolerant. Nebraska is the only state here with substantial reported yields for both crops (California and Delaware have sorghum acreage but comparatively few reported yields). Relevant fields: `crop`, `yield_anomaly_pct`, `mean_dsci`, `weeks_in_d2_plus`, `yield_status`. Check your row counts before trusting a comparison.

In [ ]:
# Your analysis here


### 3. Rainfed vs irrigated

Iowa corn is rainfed; California agriculture is heavily irrigated. Relevant fields: `precip_mm`, `precip_anomaly_pct`, `yield_anomaly_pct`, `tavg_c`, `extreme_heat_days`. The two states only share corn and wheat, and the reported-row counts are lopsided — Iowa has 2,470 reported corn rows to California's 261, and California has 385 reported wheat rows to Iowa's 103. Check the crosstab from earlier before you pick your comparison.


In [ ]:
# Your analysis here


---

### Finer resolution data

The master dataset aggregates weather and drought into one row per county-crop-year. If you want daily or weekly resolution — to build your own season windows, find sub-seasonal patterns, or look at daily extremes — load the source panels:

```python
from src.data_loader import load_daily_weather, load_weekly_drought, load_soil

weather = load_daily_weather()    # 2.5M rows: daily tmax/tmin/precip per county
drought = load_weekly_drought()   # 349K rows: weekly D0–D4 per county
soil    = load_soil()             # 253 rows: one per county (static)
```

All join on `fips`. See the data dictionary for details.